# Rule-of-three step 1 — CAC by channel for one month

Runs the whole Phase-1 pipeline for the e-commerce marketing channel
attribution project inside Google Colab.

Why Colab: BigQuery Sandbox mode + the local machine only has Python
3.9 (gcloud needs 3.10+), so we bypass the whole local-auth stack.
Colab gives us Python 3.11, `google.colab.auth.authenticate_user()`
for GCP OAuth (no ADC file, no service account key), and the
BigQuery client library preinstalled.

**Sandbox constraints respected below**: no `CREATE TABLE AS SELECT`,
no DML, no streaming inserts. Only batch LOAD jobs (`load_table_from_file` /
`load_table_from_dataframe`) and plain `SELECT` queries.

## 1. Setup

In [ ]:
# db-dtypes lets `to_dataframe()` return proper pandas dtypes for
# BigQuery DATE/TIMESTAMP columns. pyarrow is used by
# load_table_from_dataframe. bigquery itself is preinstalled on Colab.
!pip install -q google-cloud-bigquery db-dtypes pyarrow

In [ ]:
from google.colab import auth
auth.authenticate_user()
# → opens a Google OAuth dialog. Sign in with the account that owns
# the `ecommerce-channel-attribution` GCP project. After confirming,
# every google-cloud-* library in this runtime uses that identity.

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "ecommerce-channel-attribution"
LOCATION = "EU"                # dataset region — must match everything else
RAW_DATASET = "olist_raw"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print(f"client OK — project={client.project} location={LOCATION}")

## 2. Create the raw dataset (idempotent)

Creating a dataset works in Sandbox. Tables inside will auto-expire
after 60 days and can't be extended past that in sandbox — re-load if
you come back after two months.

In [ ]:
from google.api_core.exceptions import Conflict

dataset_id = f"{PROJECT_ID}.{RAW_DATASET}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = LOCATION
dataset.description = (
    "Raw Olist tables + synthetic ad_spend, loaded via batch LOAD jobs "
    "from the setup_and_cac.ipynb notebook."
)

try:
    client.create_dataset(dataset)
    print(f"created {dataset_id} in {LOCATION}")
except Conflict:
    print(f"dataset {dataset_id} already exists — nothing to do")

## 3. Upload Olist CSVs to the Colab runtime

In the Colab left sidebar → **Files** icon → **Upload** — pick these
four files from `~/Documents/ecommerce-marketing-attribution/data/olist/`:

- `olist_customers_dataset.csv`
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_order_payments_dataset.csv`

They land in `/content/` (Colab's default working directory).

Files disappear when the runtime shuts down (~90 min idle). Cheap to
re-upload; no need to persist them anywhere else for now.

In [ ]:
import os

REQUIRED = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
]
missing = [f for f in REQUIRED if not os.path.exists(f"/content/{f}")]
if missing:
    raise FileNotFoundError(f"upload these to /content/ first: {missing}")
print("all 4 CSVs present in /content/")

## 4. Load Olist CSVs into `olist_raw` (batch LOAD jobs)

In [ ]:
TABLES = {
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_customers_dataset.csv": "customers",
}

for csv_name, table_name in TABLES.items():
    table_id = f"{PROJECT_ID}.{RAW_DATASET}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    with open(f"/content/{csv_name}", "rb") as fh:
        job = client.load_table_from_file(fh, table_id, job_config=job_config)
    job.result()
    table = client.get_table(table_id)
    print(f"  loaded {csv_name} → {table_id}  ({table.num_rows:,} rows)")

## 5. Generate synthetic `ad_spend` and load it

Grain: `(channel, month)`. Deliberately COARSER than orders — joining
`ad_spend` directly onto order-level rows would fan out spend across
every order and silently inflate CAC. Downstream SQL must first
aggregate customers to `(channel, month)` before joining.

Numbers are BRL to match Olist. Seed 42 → reproducible.

In [ ]:
from datetime import date
import numpy as np
import pandas as pd

CHANNELS = ["Organic", "Google Ads", "Facebook/Instagram Ads", "Email/Referral"]
SPEND_RANGES_BRL = {
    "Organic": (500, 1_500),          # SEO/content baseline — should look best on CAC
    "Google Ads": (8_000, 15_000),
    "Facebook/Instagram Ads": (5_000, 10_000),
    "Email/Referral": (500, 2_000),
}
MONTH_START = date(2016, 9, 1)
MONTH_END = date(2018, 10, 1)
SEED = 42

def month_range(start, end):
    months = []
    y, m = start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append(date(y, m, 1))
        m += 1
        if m == 13:
            m, y = 1, y + 1
    return months

months = month_range(MONTH_START, MONTH_END)
rng = np.random.default_rng(SEED)
rows = []
for channel in CHANNELS:
    lo, hi = SPEND_RANGES_BRL[channel]
    spends = rng.uniform(lo, hi, size=len(months))
    for month, spend in zip(months, spends):
        rows.append({"channel": channel, "month": month, "spend_brl": round(float(spend), 2)})
ad_spend_df = pd.DataFrame(rows)

print(ad_spend_df.head(8).to_string(index=False))
print()
print("total spend by channel (BRL):")
print(ad_spend_df.groupby('channel')['spend_brl'].sum().round(0).to_string())

In [ ]:
table_id = f"{PROJECT_ID}.{RAW_DATASET}.ad_spend"
schema = [
    bigquery.SchemaField("channel", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("month", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("spend_brl", "NUMERIC", mode="REQUIRED"),
]
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
job = client.load_table_from_dataframe(ad_spend_df, table_id, job_config=job_config)
job.result()
print(f"loaded ad_spend → {table_id}  ({client.get_table(table_id).num_rows} rows)")

## 6. Ad hoc CAC by channel for the busiest month

CLAUDE.md §10 rule-of-three step 1: verified manually before anything
gets built on top.

Contract:
- Channel = deterministic hash of `customer_unique_id` → 4 weighted
  buckets (Organic 40 / Google Ads 25 / FB-IG 20 / Email 15).
- First-purchase month = min(order_purchase_timestamp) per
  customer_unique_id, truncated to month.
- Busiest month = month with most orders (proxy for meaningful sample).
- CAC = ad_spend(channel, month) / new customers in that (channel, month).

**Fan-out guardrail**: `new_customers` CTE aggregates to
`(channel, month)` BEFORE joining `ad_spend`. Never join `ad_spend`
directly to order-level rows.

In [ ]:
sql = '''
-- Ad hoc CAC by channel for the single busiest month in Olist.
-- CLAUDE.md §10 rule-of-three step 1: verified manually before anything
-- else gets built on top. Nothing here is persisted — SELECT only,
-- Sandbox-safe.
--
-- Contract:
--   * Channel is a deterministic hash of customer_unique_id → 4 buckets
--     with the weights from CLAUDE.md §4 (Organic 40, Google Ads 25,
--     FB/IG 20, Email/Referral 15). Same customer → always same channel.
--   * First-purchase month per customer_unique_id = the month of that
--     customer's earliest order_purchase_timestamp.
--   * "Busiest month" = the month with the most orders across the whole
--     `orders` table (proxy for meaningful sample size).
--   * CAC(channel) = ad_spend(channel, busiest_month)
--                    / (new customers acquired in that channel in that month).
--
-- Fan-out guardrail (CLAUDE.md §8): `ad_spend` is at (channel, month)
-- grain. We first aggregate customers to (channel, month) in the
-- `new_customers` CTE, then join `ad_spend` on those two keys. Never
-- join `ad_spend` directly onto `orders` or `order_items`.
--
-- Placeholders `ecommerce-channel-attribution.olist_raw` are filled in by
-- python/run_adhoc_cac.py at execution time.

WITH
-- One row per customer_unique_id with a deterministic channel bucket.
-- `customers` has one row per `customer_id`, and the same person can
-- appear under multiple `customer_id` values across repeat orders, so
-- we dedupe by customer_unique_id first — otherwise the hash would be
-- fine (deterministic) but we'd count them multiple times downstream.
channels AS (
  SELECT
    customer_unique_id,
    CASE
      WHEN MOD(ABS(FARM_FINGERPRINT(customer_unique_id)), 100) < 40 THEN 'Organic'
      WHEN MOD(ABS(FARM_FINGERPRINT(customer_unique_id)), 100) < 65 THEN 'Google Ads'
      WHEN MOD(ABS(FARM_FINGERPRINT(customer_unique_id)), 100) < 85 THEN 'Facebook/Instagram Ads'
      ELSE 'Email/Referral'
    END AS channel
  FROM `ecommerce-channel-attribution.olist_raw.customers`
  GROUP BY customer_unique_id
),

-- One row per customer_unique_id with their first-purchase month.
-- Uses order_purchase_timestamp (when the customer placed the order),
-- not the delivery or approval timestamps.
first_purchase AS (
  SELECT
    c.customer_unique_id,
    DATE_TRUNC(DATE(MIN(o.order_purchase_timestamp)), MONTH) AS first_purchase_month
  FROM `ecommerce-channel-attribution.olist_raw.orders` o
  JOIN `ecommerce-channel-attribution.olist_raw.customers` c
    ON o.customer_id = c.customer_id
  GROUP BY c.customer_unique_id
),

-- The single busiest month across all orders. Kept as its own CTE so
-- the choice is visible in the output (not hidden behind a scalar
-- subquery).
busiest_month AS (
  SELECT DATE_TRUNC(DATE(order_purchase_timestamp), MONTH) AS month
  FROM `ecommerce-channel-attribution.olist_raw.orders`
  GROUP BY month
  ORDER BY COUNT(*) DESC
  LIMIT 1
),

-- Aggregate new customers to (channel, month) grain BEFORE joining to
-- ad_spend — CLAUDE.md §8 fan-out rule.
new_customers AS (
  SELECT
    ch.channel,
    fp.first_purchase_month AS month,
    COUNT(*) AS new_customer_count
  FROM first_purchase fp
  JOIN channels ch USING (customer_unique_id)
  WHERE fp.first_purchase_month = (SELECT month FROM busiest_month)
  GROUP BY ch.channel, fp.first_purchase_month
)

SELECT
  nc.month                                    AS busiest_month,
  nc.channel,
  nc.new_customer_count,
  s.spend_brl,
  ROUND(s.spend_brl / nc.new_customer_count, 2) AS cac_brl
FROM new_customers nc
JOIN `ecommerce-channel-attribution.olist_raw.ad_spend` s
  ON s.channel = nc.channel
 AND s.month = nc.month
ORDER BY cac_brl ASC;
'''
df = client.query(sql).result().to_dataframe()
df

In [ ]:
print(f"total new customers this month: {int(df['new_customer_count'].sum())}")
print(f"total spend this month (BRL):   {float(df['spend_brl'].sum()):,.2f}")

## 7. Manual verification checklist

Before promoting this into a persisted mart:

- [ ] Does the busiest month look reasonable? (Olist's peak is
      typically **2017-11** = Black Friday.)
- [ ] Do channel counts roughly match the 40/25/20/15 weights?
- [ ] Is Organic's CAC very low (matches our story about "cheap by
      CAC but that's misleading")?
- [ ] Do the row count and total spend look sane?

If all four are yes → we've verified step 1. Only then does the same
logic get moved into a dbt mart (Phase 2, once billing is on).